# Module 11 Guided Lab: Final Data Checks, Data Dictionary, and ABT Documentation

**BAN 6003: Data Management and Analytics Integration**

This module is the final preparation step before we apply analytics to an ABT.

We are not learning many new Pandas or SQL functions this week. Instead, we are using the skills we already learned to ask:

> Is this ABT ready for modeling, handoff, and business interpretation?

A good ABT should be checked, documented, and understandable to someone other than the person who built it.

## Lab Learning Goals

By the end of this lab, you should be able to:

1. Conduct final sanity checks on a prepared ABT.
2. Check missing values, impossible values, duplicate entity rows, and target variable issues.
3. Create a simple data dictionary for an ABT.
4. Write a transformation and integration summary.
5. Prepare a data handoff package.
6. Apply this same process to your semester project ABT.

Starting next module, we will apply analytics and modeling. That means this is the moment to slow down and make sure the ABT is trustworthy.

## Business Scenario

Imagine you are handing your ABT to another analyst or to your future self.

That person needs to understand:

- what one row means
- where the columns came from
- what transformations were applied
- what the target variable means
- what missing values or limitations still exist
- whether the table is safe to use for modeling

If they cannot understand these points, the ABT is not ready for handoff.

## 0. Setup

This lab can work in two ways:

1. If you completed Module 10, you can use your own preliminary ABT.
2. If not, the notebook will create a small demonstration ABT from `nycflights13`.

For the demo version, we will build a route-level ABT with one row per origin-destination route.

### Package setup note

If you are using **GitHub Codespaces**, the required packages should already be installed from `requirements.txt` when the Codespace is created. You usually do not need to run any `%pip install` command.

If you are running the repository on your **local computer** and an import fails, uncomment and run the `%pip install` line(s) in the setup cell below, then rerun the imports.



In [ ]:
# Codespaces: nycflights13 is installed from requirements.txt.
# Local only: if the import below fails, uncomment and run this line:
# %pip install nycflights13

import nycflights13
import pandas as pd
import numpy as np
from pathlib import Path


In [ ]:
flights = nycflights13.flights.copy()
airports = nycflights13.airports.copy()
airlines = nycflights13.airlines.copy()

## 1. Build a Demo ABT

For this demonstration, we will create a route-level ABT.

**Unit of analysis:** one row per origin-destination route.

This ABT is not meant to be perfect. It is a realistic practice table for final checks and documentation.

In [ ]:
route_abt = (
    flights
    .groupby(["origin", "dest"])
    .agg(
        flight_count=("flight", "count"),
        avg_dep_delay=("dep_delay", "mean"),
        avg_arr_delay=("arr_delay", "mean"),
        max_arr_delay=("arr_delay", "max"),
        avg_distance=("distance", "mean"),
        avg_air_time=("air_time", "mean")
    )
    .reset_index()
)

route_abt.head()

Add destination airport name to make the table more understandable.

This is a left merge because the route-level ABT is our main table and we want to preserve its rows.

In [ ]:
dest_airports = airports[["faa", "name", "lat", "lon"]].rename(
    columns={
        "faa": "dest",
        "name": "dest_airport_name",
        "lat": "dest_lat",
        "lon": "dest_lon"
    }
)

route_abt = route_abt.merge(dest_airports, how="left", on="dest")

route_abt.head()

## 2. Final Sanity Check 1: Row Meaning and Shape

Before checking values, first confirm what one row means.

For this ABT:

> One row should represent one origin-destination route.

The key columns should be `origin` and `dest`.

In [ ]:
route_abt.shape

In [ ]:
route_abt[["origin", "dest"]].head()

### Check duplicate entity rows

If one row should represent one route, then each origin-destination pair should appear only once.

In [ ]:
route_abt[["origin", "dest"]].duplicated().sum()

If this number is greater than zero, the ABT has duplicate entity rows and should not be used for modeling until the issue is resolved.

### Your Turn 1

Write one sentence explaining what one row means in this ABT.

Then identify the key column or columns that define one row.

**Your answer:**  
Type your answer here.

## 3. Final Sanity Check 2: Missing Values

Missing values are not always wrong, but they must be understood.

For an ABT, missing values can affect:

- modeling
- interpretation
- business recommendations
- whether a column should be used at all

In [ ]:
missing_summary = route_abt.isna().sum().reset_index()
missing_summary.columns = ["column_name", "missing_count"]
missing_summary["missing_percent"] = missing_summary["missing_count"] / len(route_abt)

missing_summary.sort_values("missing_count", ascending=False)

### Questions to ask

For any column with missing values:

- Is this missingness expected?
- Did it come from the raw data?
- Did it come from an integration step?
- Should the column be kept, dropped, or imputed later?
- Should the limitation be documented?

This module is about checking and documenting, not making every dataset perfect.

### Your Turn 2

Identify one column with missing values, or confirm that no important columns have missing values.

Write one short note explaining what you would do next.

**Your answer:**  
Type your answer here.

## 4. Final Sanity Check 3: Impossible or Suspicious Values

Some values may be technically non-missing but still suspicious.

Examples:

- negative flight counts
- negative distances
- impossible average air times
- unusually large delay values

Not every extreme value is wrong. But it should be noticed.

In [ ]:
route_abt.describe()

In [ ]:
# Basic impossible-value checks
checks = {
    "negative_flight_count": (route_abt["flight_count"] < 0).sum(),
    "zero_or_negative_distance": (route_abt["avg_distance"] <= 0).sum(),
    "zero_or_negative_air_time": (route_abt["avg_air_time"] <= 0).sum(),
}

checks

### Check extreme values

Now sort to inspect routes with very high average arrival delay.

In [ ]:
route_abt.sort_values("avg_arr_delay", ascending=False).head(10)

Extreme values do not automatically mean errors. They may be real business signals. But they should be reviewed before modeling.

### Your Turn 3

Choose one numeric column and inspect its minimum, maximum, and average.

Write one sentence explaining whether the values look reasonable.

In [ ]:
# Your Turn 3
# Choose one numeric column and inspect min, max, and mean.

**Your answer:**  
Type your answer here.

## 5. Final Sanity Check 4: Target Variable Issues

If the ABT will be used for modeling, the target variable must be clearly defined.

For this demo ABT, suppose our target is:

> average arrival delay by route

That means `avg_arr_delay` is the outcome we may want to explain or predict.

In [ ]:
route_abt["avg_arr_delay"].isna().sum()

In [ ]:
route_abt["avg_arr_delay"].describe()

### Questions to ask about the target

- What exactly does the target measure?
- Is the target missing for some rows?
- Is the target available at the time of prediction?
- Are there extreme values that should be reviewed?
- Would this target support the business question?

These questions help prevent modeling mistakes later.

### Your Turn 4

For your project ABT, identify your likely target variable.

If you do not know yet, identify one possible target variable.

Write 2–3 sentences explaining what it measures and whether it has any concerns.

**Your answer:**  
Type your answer here.

## 6. Create a Data Dictionary

A data dictionary explains what each column means.

At minimum, include:

- column name
- data type
- description
- source
- transformation or derivation
- notes or concerns

A data dictionary is not busywork. It is what makes your dataset usable by someone else.

In [ ]:
data_dictionary = pd.DataFrame({
    "column_name": route_abt.columns,
    "data_type": [str(route_abt[col].dtype) for col in route_abt.columns],
    "description": "",
    "source": "",
    "transformation_or_derivation": "",
    "notes_or_concerns": ""
})

data_dictionary

Now fill in several rows manually as examples.

In a real project, you should complete this for every final ABT column.

In [ ]:
data_dictionary.loc[data_dictionary["column_name"] == "origin", ["description", "source", "transformation_or_derivation"]] = [
    "Origin airport code for the route", "flights table", "Used as part of route-level grouping key"
]

data_dictionary.loc[data_dictionary["column_name"] == "dest", ["description", "source", "transformation_or_derivation"]] = [
    "Destination airport code for the route", "flights table", "Used as part of route-level grouping key"
]

data_dictionary.loc[data_dictionary["column_name"] == "flight_count", ["description", "source", "transformation_or_derivation"]] = [
    "Number of flights observed for the route", "flights table", "Count of flight records grouped by origin and destination"
]

data_dictionary.loc[data_dictionary["column_name"] == "avg_arr_delay", ["description", "source", "transformation_or_derivation"]] = [
    "Average arrival delay for flights on the route", "flights table", "Mean of arr_delay grouped by origin and destination"
]

data_dictionary.loc[data_dictionary["column_name"] == "dest_airport_name", ["description", "source", "transformation_or_derivation"]] = [
    "Full name of the destination airport", "airports table", "Joined from airports using dest = faa"
]

data_dictionary

### Your Turn 5

Choose three additional columns and complete the data dictionary fields for them.

You can edit the table directly using `.loc[]`, or create your own markdown table below.

In [ ]:
# Your Turn 5
# Add documentation for at least three more columns.

## 7. Transformation and Integration Summary

The data dictionary documents columns. The transformation summary documents the process.

A good transformation summary answers:

- What raw data was used?
- What cleaning or filtering was applied?
- What transformations were created?
- What aggregations changed the unit of analysis?
- What tables were joined?
- What checks were performed?
- What assumptions or limitations remain?

### Example Transformation Summary

For this demo ABT:

1. Started with the `flights` table from `nycflights13`.
2. Aggregated flight-level records to one row per origin-destination route.
3. Calculated flight count, average departure delay, average arrival delay, maximum arrival delay, average distance, and average air time.
4. Joined destination airport name and location from the `airports` table using `dest = faa`.
5. Checked duplicate route keys using `origin` and `dest`.
6. Checked missing values across ABT columns.
7. Reviewed basic impossible values and extreme delay values.
8. The target variable for later modeling may be `avg_arr_delay`, but this choice should be justified by the business question.

### Your Turn 6

Write a transformation and integration summary for your own project ABT.

If your project ABT is not ready yet, write a draft summary based on your planned workflow.

**Your transformation summary:**  
Type your answer here.

## 8. Prepare a Handoff Package

A basic ABT handoff package should include:

1. Final ABT file
2. Data dictionary
3. Transformation and integration summary
4. Validation checklist
5. README or short handoff note

For this lab, we will export the demo ABT and data dictionary.

In [ ]:
output_dir = Path("module11_outputs")
output_dir.mkdir(exist_ok=True)

route_abt.to_csv(output_dir / "route_abt_demo.csv", index=False)
data_dictionary.to_csv(output_dir / "route_abt_data_dictionary_demo.csv", index=False)

print("Files saved to:", output_dir)

### Create a simple validation checklist

This checklist records whether the most important checks were completed.

In [ ]:
validation_checklist = pd.DataFrame({
    "check": [
        "Unit of analysis is clearly defined",
        "Key columns are identified",
        "Duplicate entity rows checked",
        "Missing values checked",
        "Impossible or suspicious values reviewed",
        "Target variable reviewed",
        "Data dictionary started",
        "Transformation summary written"
    ],
    "status": [
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "In progress",
        "In progress"
    ],
    "notes": [
        "One row per origin-destination route",
        "origin + dest",
        "Checked with duplicated().sum()",
        "Checked with isna().sum()",
        "Reviewed basic numeric ranges",
        "avg_arr_delay reviewed as possible target",
        "Example rows completed; project version should be completed fully",
        "Example provided; project version should be completed fully"
    ]
})

validation_checklist

In [ ]:
validation_checklist.to_csv(output_dir / "route_abt_validation_checklist_demo.csv", index=False)

## 9. Apply This to Your Project

This week, you should apply the same process to your semester project.

For your project, prepare:

1. A clear statement of your ABT unit of analysis.
2. A final validation checklist.
3. A data dictionary for your ABT.
4. A transformation and integration summary.
5. A clean handoff folder in GitHub.

This work prepares you for the next module, where we begin applying analytics to the ABT.

## 10. Final Reflection

Write 5–7 sentences answering:

1. What does one row mean in your project ABT?
2. What are your key columns?
3. What is your likely target variable?
4. What was the most important validation check?
5. Which column needs the clearest explanation in the data dictionary?
6. What still needs to be improved before modeling?

**Your reflection:**  
Type your answer here.

## 11. Save Your Work

Before submitting:

1. Save this notebook.
2. Confirm your output files were created.
3. Complete all Your Turn sections.
4. Complete the final reflection.
5. Commit and push your work to GitHub.

Suggested commands:

```bash
git add .
git commit -m "Complete Module 11 ABT documentation lab"
git push
```